# 15 — Robustness and Validation

## Goal

Check whether the pooled-model conclusions remain similar under reasonable changes.

Models from Notebook 09:

- Dummy Classifier
- Logistic Regression
- Random Forest
- XGBoost
- supplementary LSTM

The checks are model-specific: deterministic Dummy is not seed-tested, while LSTM uses lookback sensitivity instead of SMA/EMA sensitivity. Main metrics are Balanced Accuracy and Macro F1.

## 1. Setup and Load Data

We use the same dataset, target, feature exclusions, model settings, and temporal split as the pooled model.

- Train: 2020–2024
- Test: 2025
- Baseline target threshold: ±1.5%

In [1]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, f1_score
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_YEAR = 2025
TARGET = "target_5d"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
pd.set_option("display.float_format", "{:.4f}".format)


In [2]:
DATA_PATH = "/kaggle/input/notebooks/phyothaw/06-target-and-model-dataset-ipynb/model_ready_complete_case_2020_2025.csv"

data_df = pd.read_csv(DATA_PATH)
data_df["date"] = pd.to_datetime(data_df["date"])

# Same missing-GDELT treatment as the pooled model
data_df["gdelt_tone_missing"] = data_df["gdelt_company_tone"].isna().astype(int)
data_df["gdelt_company_tone"] = data_df["gdelt_company_tone"].fillna(0)

print("Shape:", data_df.shape)
print("Stocks:", sorted(data_df["ticker"].unique()))
print("Date range:", data_df["date"].min(), "to", data_df["date"].max())

Shape: (7370, 42)
Stocks: ['BRK-B', 'CVX', 'GE', 'MSFT', 'NVDA']
Date range: 2020-01-13 00:00:00 to 2025-12-22 00:00:00


In [3]:
DROP_COLUMNS = [
    "ticker", "date",
    "future_close_5d", "forward_return_5d",
    "movement_5d", "target_5d",
    "Dividends", "Stock Splits",
    "daily_return_check",
]

FEATURE_COLUMNS = [c for c in data_df.columns if c not in DROP_COLUMNS]

train_df = data_df[data_df["date"].dt.year < TEST_YEAR].copy()
test_df = data_df[data_df["date"].dt.year == TEST_YEAR].copy()
stocks = sorted(data_df["ticker"].unique())

print("Features:", len(FEATURE_COLUMNS))
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))


Features: 33
Train rows: 6240
Test rows: 1130


## 2. No-leakage and data-integrity checks

These checks apply to all five models. LSTM sequences are constructed separately for each ticker later in the notebook.


In [4]:
forbidden = ["future_close_5d", "forward_return_5d", "movement_5d", "target_5d"]

checks = {
    "Future/target columns excluded": not any(c in FEATURE_COLUMNS for c in forbidden),
    "Ticker excluded from predictors": "ticker" not in FEATURE_COLUMNS,
    "Training ends before testing": train_df["date"].max() < test_df["date"].min(),
    "2025 used only for testing": test_df["date"].dt.year.eq(TEST_YEAR).all(),
    "No NaN in model features": data_df[FEATURE_COLUMNS].isna().sum().sum() == 0,
    "Target contains only classes 0, 1, 2": set(data_df[TARGET].unique()).issubset({0, 1, 2}),
}

validation_checks = pd.DataFrame({
    "check": checks.keys(),
    "status": ["PASS" if value else "FAIL" for value in checks.values()],
})

display(validation_checks)
assert validation_checks["status"].eq("PASS").all(), "A validation check failed."


,check,status
0,Future/target columns excluded,PASS
1,Ticker excluded from predictors,PASS
2,Training ends before testing,PASS
3,2025 used only for testing,PASS
4,No NaN in model features,PASS
5,"Target contains only classes 0, 1, 2",PASS


## 3. Four tabular models

One helper keeps preprocessing and evaluation consistent. Dummy is included as the non-informative benchmark.


In [5]:
def run_tabular_models(train, test, features, target_col, seed=RANDOM_STATE):
    X_train = train[features]
    y_train = train[target_col]
    X_test = test[features]
    y_test = test[target_col]
    rows = []

    models = {
        "Dummy Classifier": DummyClassifier(strategy="most_frequent"),
        "Random Forest": RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=seed, n_jobs=-1
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            objective="multi:softprob", num_class=3, eval_metric="mlogloss",
            random_state=seed, n_jobs=-1,
        ),
    }

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        rows.append({
            "model": name,
            "balanced_accuracy": balanced_accuracy_score(y_test, pred),
            "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
        })

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    lr = LogisticRegression(
        penalty="l2", class_weight="balanced", max_iter=2000, random_state=seed
    )
    lr.fit(X_train_scaled, y_train)
    pred = lr.predict(X_test_scaled)
    rows.append({
        "model": "Logistic Regression",
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
    })

    return pd.DataFrame(rows)


baseline_results = run_tabular_models(
    train_df, test_df, FEATURE_COLUMNS, TARGET
).sort_values("balanced_accuracy", ascending=False)

display(baseline_results)


,model,balanced_accuracy,macro_f1
3,Logistic Regression,0.3899,0.3818
1,Random Forest,0.3707,0.3543
2,XGBoost,0.3526,0.3379
0,Dummy Classifier,0.3333,0.1819


## 4. Random-seed stability for Random Forest and XGBoost

Dummy is deterministic and does not need a seed test. Logistic Regression is effectively deterministic with this configuration. LSTM is seed-tested separately.


In [6]:
SEEDS = [42, 123, 456, 789, 2026]
seed_rows = []

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET]
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET]

for seed in SEEDS:
    seed_models = {
        "Random Forest": RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=seed, n_jobs=-1
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            objective="multi:softprob", num_class=3, eval_metric="mlogloss",
            random_state=seed, n_jobs=-1,
        ),
    }

    for name, model in seed_models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        seed_rows.append({
            "model": name,
            "seed": seed,
            "balanced_accuracy": balanced_accuracy_score(y_test, pred),
            "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
        })

seed_results = pd.DataFrame(seed_rows)
seed_summary = seed_results.groupby("model").agg(
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
    balanced_accuracy_std=("balanced_accuracy", "std"),
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_std=("macro_f1", "std"),
).reset_index()

display(seed_summary)


,model,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std
0,Random Forest,0.3617,0.0059,0.3338,0.0123
1,XGBoost,0.3592,0.0054,0.3415,0.0048


## 5. Target-threshold sensitivity

The four tabular models are evaluated at ±1.0%, ±1.5%, and ±2.0%. Dummy is included because changing the threshold changes the class distribution. Repeating the complete LSTM temporal-validation pipeline at every threshold is outside this simple screening notebook and is reported as a limitation.


In [7]:
THRESHOLDS = {"1.0%": 0.010, "1.5%": 0.015, "2.0%": 0.020}
threshold_rows = []
class_rows = []

for label, threshold in THRESHOLDS.items():
    temp = data_df.copy()
    temp["robust_target"] = np.select(
        [temp["forward_return_5d"] < -threshold,
         temp["forward_return_5d"] > threshold],
        [0, 2],
        default=1,
    )

    threshold_train = temp[temp["date"].dt.year < TEST_YEAR].copy()
    threshold_test = temp[temp["date"].dt.year == TEST_YEAR].copy()

    result = run_tabular_models(
        threshold_train, threshold_test, FEATURE_COLUMNS, "robust_target"
    )
    result["threshold"] = label
    threshold_rows.append(result)

    proportions = threshold_test["robust_target"].value_counts(normalize=True)
    class_rows.append({
        "threshold": label,
        "down": proportions.get(0, 0),
        "neutral": proportions.get(1, 0),
        "up": proportions.get(2, 0),
    })

threshold_results = pd.concat(threshold_rows, ignore_index=True)
threshold_pivot = threshold_results.pivot(
    index="threshold", columns="model", values="balanced_accuracy"
).reindex(THRESHOLDS.keys())

display(pd.DataFrame(class_rows))
display(threshold_pivot)


,threshold,down,neutral,up
0,1.0%,0.3035,0.2549,0.4416
1,1.5%,0.2496,0.3752,0.3752
2,2.0%,0.2186,0.4699,0.3115


model,Dummy Classifier,Logistic Regression,Random Forest,XGBoost
threshold,,,,
1.0%,0.3333,0.3751,0.3479,0.3498
1.5%,0.3333,0.3899,0.3707,0.3526
2.0%,0.3333,0.4041,0.3696,0.4007


## 6. Short-window feature sensitivity for tabular models

This compares the original features with the same features plus SMA_5 and EMA_5.


In [8]:
window_df = data_df.copy()
window_df["SMA_5"] = window_df.groupby("ticker")["Close"].transform(
    lambda values: values.rolling(5).mean()
)
window_df["EMA_5"] = window_df.groupby("ticker")["Close"].transform(
    lambda values: values.ewm(span=5, adjust=False).mean()
)
window_df = window_df.dropna(subset=["SMA_5", "EMA_5"]).copy()

window_train = window_df[window_df["date"].dt.year < TEST_YEAR].copy()
window_test = window_df[window_df["date"].dt.year == TEST_YEAR].copy()

base_window = run_tabular_models(
    window_train, window_test, FEATURE_COLUMNS, TARGET
)
base_window["feature_set"] = "Baseline"

extended_window = run_tabular_models(
    window_train, window_test, FEATURE_COLUMNS + ["SMA_5", "EMA_5"], TARGET
)
extended_window["feature_set"] = "Baseline + SMA_5 + EMA_5"

window_results = pd.concat([base_window, extended_window], ignore_index=True)
window_pivot = window_results.pivot(
    index="model", columns="feature_set", values="balanced_accuracy"
)
window_pivot["change"] = (
    window_pivot["Baseline + SMA_5 + EMA_5"] - window_pivot["Baseline"]
)

display(window_pivot)


feature_set,Baseline,Baseline + SMA_5 + EMA_5,change
model,,,
Dummy Classifier,0.3333,0.3333,0.0000
Logistic Regression,0.3993,0.3958,-0.0036
Random Forest,0.3448,0.3467,0.0020
XGBoost,0.3624,0.3581,-0.0043


## 7. Simple LSTM robustness

This section is self-contained. It builds 30-day sequences separately for every ticker, fits the scaler only on 2020–2024, and evaluates 2025. A fixed 15-epoch budget is used for every run so that only the seed or lookback changes.

Three seeds are sufficient for this computationally heavier supplementary model. Lookbacks of 20, 30, and 60 days replace the SMA/EMA test because sequence length is the more relevant LSTM assumption.


In [9]:
import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import LSTM, Dense, Dropout

LSTM_EPOCHS = 15
LSTM_SEEDS = [42, 123, 456]
LOOKBACKS = [20, 30, 60]


def build_lstm_sequences(target_df, features, target_col, lookback, context_df=None):
    X_sequences, y_values = [], []

    for ticker in stocks:
        target_stock = target_df[target_df["ticker"] == ticker].sort_values("date")

        if context_df is not None:
            context_stock = (
                context_df[context_df["ticker"] == ticker]
                .sort_values("date")
                .tail(lookback)
            )
            combined = pd.concat([context_stock, target_stock], ignore_index=True)
            target_start = len(context_stock)
        else:
            combined = target_stock.reset_index(drop=True)
            target_start = lookback

        for i in range(target_start, len(combined)):
            if i >= lookback:
                X_sequences.append(combined.iloc[i-lookback:i][features].to_numpy())
                y_values.append(combined.iloc[i][target_col])

    return np.asarray(X_sequences, dtype=np.float32), np.asarray(y_values, dtype=np.int32)


def create_lstm_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(64),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(3, activation="softmax"),
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# Scale without using any 2025 information.
lstm_scaler = StandardScaler()
lstm_train = train_df.copy()
lstm_test = test_df.copy()
lstm_train[FEATURE_COLUMNS] = lstm_scaler.fit_transform(train_df[FEATURE_COLUMNS])
lstm_test[FEATURE_COLUMNS] = lstm_scaler.transform(test_df[FEATURE_COLUMNS])

print("TensorFlow:", tf.__version__)


TensorFlow: 2.20.0


In [10]:
# LSTM random-seed stability using the baseline 30-day lookback.
x_lstm_train, y_lstm_train = build_lstm_sequences(
    lstm_train, FEATURE_COLUMNS, TARGET, lookback=30
)
x_lstm_test, y_lstm_test = build_lstm_sequences(
    lstm_test, FEATURE_COLUMNS, TARGET, lookback=30, context_df=lstm_train
)

lstm_seed_rows = []

for seed in LSTM_SEEDS:
    tf.keras.backend.clear_session()
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    model = create_lstm_model(x_lstm_train.shape[1:])
    model.fit(
        x_lstm_train, y_lstm_train,
        epochs=LSTM_EPOCHS, batch_size=32,
        shuffle=False, verbose=0,
    )
    pred = np.argmax(model.predict(x_lstm_test, verbose=0), axis=1)
    lstm_seed_rows.append({
        "model": "LSTM",
        "seed": seed,
        "balanced_accuracy": balanced_accuracy_score(y_lstm_test, pred),
        "macro_f1": f1_score(y_lstm_test, pred, average="macro", zero_division=0),
    })

lstm_seed_results = pd.DataFrame(lstm_seed_rows)
display(lstm_seed_results)
display(lstm_seed_results[["balanced_accuracy", "macro_f1"]].agg(["mean", "std"]))


I0000 00:00:1786992841.710527    3363 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786992841.712616    3363 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


,model,seed,balanced_accuracy,macro_f1
0,LSTM,42,0.3773,0.3683
1,LSTM,123,0.3467,0.3110
2,LSTM,456,0.3544,0.3226


,balanced_accuracy,macro_f1
mean,0.3595,0.3340
std,0.0159,0.0303


In [11]:
# LSTM lookback sensitivity using one fixed seed.
lstm_lookback_rows = []

for lookback in LOOKBACKS:
    x_train_seq, y_train_seq = build_lstm_sequences(
        lstm_train, FEATURE_COLUMNS, TARGET, lookback=lookback
    )
    x_test_seq, y_test_seq = build_lstm_sequences(
        lstm_test, FEATURE_COLUMNS, TARGET,
        lookback=lookback, context_df=lstm_train
    )

    tf.keras.backend.clear_session()
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    tf.keras.utils.set_random_seed(RANDOM_STATE)

    model = create_lstm_model(x_train_seq.shape[1:])
    model.fit(
        x_train_seq, y_train_seq,
        epochs=LSTM_EPOCHS, batch_size=32,
        shuffle=False, verbose=0,
    )
    pred = np.argmax(model.predict(x_test_seq, verbose=0), axis=1)
    lstm_lookback_rows.append({
        "lookback": lookback,
        "balanced_accuracy": balanced_accuracy_score(y_test_seq, pred),
        "macro_f1": f1_score(y_test_seq, pred, average="macro", zero_division=0),
    })

lstm_lookback_results = pd.DataFrame(lstm_lookback_rows)
display(lstm_lookback_results)


,lookback,balanced_accuracy,macro_f1
0,20,0.3429,0.3392
1,30,0.3344,0.3020
2,60,0.3571,0.3557


## 8. Robustness summary

Interpret the evidence rather than forcing every model through an unsuitable test:

- All models must pass the common leakage and data-integrity checks.
- Dummy provides a sanity-check benchmark but is not seed-tested.
- Random Forest, XGBoost, and LSTM should show small variation across seeds.
- Threshold changes should not radically reverse the tabular-model conclusion.
- SMA/EMA changes should have only a small effect on tabular models.
- LSTM results should remain broadly similar across reasonable lookbacks.

The LSTM threshold experiment is omitted to keep this notebook computationally manageable; this should be reported as a limitation rather than hidden.


In [12]:
summary = pd.DataFrame([
    {
        "check": "Common validation checks",
        "evidence": f"{validation_checks['status'].eq('PASS').sum()} of {len(validation_checks)} passed",
    },
    {
        "check": "RF/XGBoost maximum seed BA SD",
        "evidence": f"{seed_summary['balanced_accuracy_std'].max():.4f}",
    },
    {
        "check": "LSTM seed BA SD",
        "evidence": f"{lstm_seed_results['balanced_accuracy'].std():.4f}",
    },
    {
        "check": "Largest tabular window BA change",
        "evidence": f"{window_pivot['change'].abs().max():.4f}",
    },
    {
        "check": "LSTM lookback BA range",
        "evidence": f"{lstm_lookback_results['balanced_accuracy'].max() - lstm_lookback_results['balanced_accuracy'].min():.4f}",
    },
])

display(summary)
print("Best tabular model at each threshold:")
display(threshold_pivot.idxmax(axis=1).rename("best_model").to_frame())


,check,evidence
0,Common validation checks,6 of 6 passed
1,RF/XGBoost maximum seed BA SD,0.0059
2,LSTM seed BA SD,0.0159
3,Largest tabular window BA change,0.0043
4,LSTM lookback BA range,0.0227


Best tabular model at each threshold:


,best_model
threshold,
1.0%,Logistic Regression
1.5%,Logistic Regression
2.0%,Logistic Regression


In [13]:
# Save robustness tables so Notebook 16 can load them
seed_results.to_csv("/kaggle/working/robustness_seed_stability.csv", index=False)
threshold_results.to_csv("/kaggle/working/robustness_threshold_sensitivity.csv", index=False)
window_results.to_csv("/kaggle/working/robustness_window_sensitivity.csv", index=False)

print("Saved:")
print("/kaggle/working/robustness_seed_stability.csv")
print("/kaggle/working/robustness_threshold_sensitivity.csv")
print("/kaggle/working/robustness_window_sensitivity.csv")

Saved:
/kaggle/working/robustness_seed_stability.csv
/kaggle/working/robustness_threshold_sensitivity.csv
/kaggle/working/robustness_window_sensitivity.csv
